# DINO — Time Series Self-Supervised Pretraining + Forecasting

DWT student-teacher with soft-threshold teacher views and stochastic student augmentations.

**Pipeline:** Pretrain on ETT → Frozen-encoder linear forecasting head fine-tune → Test

## 1 — Mount Drive & set project root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Change this to wherever you uploaded the allthree folder ──────────────────
PROJECT_ROOT = '/content/drive/MyDrive/allthree'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 2 — Choose dataset

In [ ]:
# Pretrain on Monash (all .tsf files) — controlled by pretrain_on_monash in config.py
# Set FORECAST_DATASET for the downstream task.
FORECAST_DATASET = 'etth1'   # ETTh1 for forecasting evaluation

from dataset_registry import get_dataset_info
import pandas as pd

ds = get_dataset_info(FORECAST_DATASET)
df = pd.read_csv(ds['csv_path'], nrows=0)
print(f'Forecast → {FORECAST_DATASET}  {ds["c_in"]} vars   {ds["csv_path"]}')

## 3 — Install dependencies

In [ ]:
!pip install PyWavelets --quiet

import torch, sklearn, pywt
print(f'torch   {torch.__version__}')
print(f'sklearn {sklearn.__version__}')
print(f'pywt    {pywt.__version__}')
print(f'GPU     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT available — go to Runtime > Change runtime type > T4 GPU"}')

## 4 — Verify data

In [ ]:
ds = get_dataset_info(FORECAST_DATASET)
df = pd.read_csv(ds['csv_path'])
print(f'── Forecast: {FORECAST_DATASET} ──')
print(f'  Rows    : {len(df)}')
print(f'  Columns : {df.columns.tolist()}')
print(df.head(2))

## 5 — (Optional) Tune config before running

Edit `TSDINOALT 4/config.py` on Drive, or override specific keys here:

In [ ]:
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'TSDINOALT 4'))
import importlib, config as dino_cfg_mod
importlib.reload(dino_cfg_mod)          # ensure fresh load after Drive sync

# dino_cfg_mod.config['epochs']       = 20    # pretrain epochs
# dino_cfg_mod.config['saveckp_freq'] = 5

print('DINO config snapshot:')
for k in ['epochs', 'out_dim', 'embed_dim', 'patch_len', 'num_patches',
          'batch_size_per_gpu', 'pretrain_on_monash', 'monash_data_dir']:
    print(f'  {k:30s} = {dino_cfg_mod.config[k]}')

## 6 — Run DINO (pretrain → forecasting)

In [ ]:
from Train_and_downstream import run

run(
    model            = 'dino',
    forecast_dataset = FORECAST_DATASET,
    skip_train       = False,
)

## 7 — Forecasting only (load existing checkpoint)

Set `skip_train=True` and update `path_num` in config to load a saved checkpoint.

In [ ]:
# dino_cfg_mod.config['path_num'] = 10   # checkpoint epoch to load
# run(model='dino', pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=True)